# Notebook 01 — Exploratory Data Analysis (Raw Data)

**Goal:** Understand the dataset *before* any transformation.  
All plots reflect the raw, unprocessed distributions — the ones that belong in the EDA chapter of the academic report.

**Key Principle:** EDA on RAW data, not transformed data. (CRISP-DM Phase 2)

**Input:** `data/raw/big_startup_secsees_dataset.csv`  
**Output:** No files saved — read-only exploration. Plots saved to `results/eda/`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import yaml
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Load config — single source of truth for all parameters
with open('../configs/params.yaml', encoding='utf-8') as f:
    CFG = yaml.safe_load(f)

RANDOM_SEED = CFG['random_seed']
os.makedirs('results/eda', exist_ok=True)
print('Setup complete.')

## 1. Load Raw Data

We load the CSV and perform **minimal** fixes so we can explore meaningfully:
- Fix `funding_total_usd` from string to float (the '-' placeholder blocks all numeric analysis)
- Parse dates to datetime (needed for temporal analysis)

We do NOT impute, encode, scale, or drop any rows here. That's nb02's job.

In [ ]:
RAW_PATH = CFG['paths']['raw_data']

df = pd.read_csv(RAW_PATH, skipinitialspace=True)
df.columns = df.columns.str.strip()

# Fix funding: replace placeholder '-' with NaN, cast to float
df['funding_total_usd'] = (
    df['funding_total_usd'].astype(str).str.strip()
    .replace('-', np.nan).replace('nan', np.nan)
)
df['funding_total_usd'] = pd.to_numeric(df['funding_total_usd'], errors='coerce')

# Parse dates (keep NaT — no imputation in EDA)
for col in ['founded_at', 'first_funding_at', 'last_funding_at']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 2. Basic Overview

Understanding types is the first step of any EDA. The key question for each column:
*What type should it be vs what type is it?*

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Numeric Summary ===')
df.describe(include='all').T

## 3. Missing Values Analysis

Not all missing values are the same. We classify each column's missingness:
- **founded_at (23%):** MAR — older startups less likely to have dates recorded
- **country_code (10.5%):** Probably MCAR — random gaps in data collection
- **funding_total_usd:** Mix of true NaN and '-' (which we converted) — likely MNAR (unfunded startups)

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'percent': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('percent', ascending=False)
print(missing_df.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
missing_df['percent'].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values per Column')
for i, v in enumerate(missing_df['percent']):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../results/eda/missing_values.png', bbox_inches='tight')
plt.show()

## 4. Target Variable Distribution

This is the most critical plot. It reveals:
1. **Severe class imbalance** -- operating dominates at ~80%
2. **All 4 classes are clean** -- no corrupted rows in the CSV file
3. The imbalance means accuracy is a misleading metric -> we use **macro F1** instead

In [ ]:
# Show status distribution
print('Status value counts:')
print(df['status'].value_counts())
print()
print(f'Total valid rows: {len(df):,}')
print('All status values are clean in the CSV file.')

# Plot valid statuses
df_valid = df.copy()
status_counts = df_valid['status'].value_counts()
status_pct = (status_counts / len(df_valid) * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']

status_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Startup Status -- Count')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=9)

axes[1].pie(status_counts, labels=status_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=140, pctdistance=0.8)
axes[1].set_title('Startup Status -- Proportion')

plt.suptitle('Target Variable: Class Imbalance is Significant', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../results/eda/target_distribution.png', bbox_inches='tight')
plt.show()

print(f'\nClass distribution:')
for status, pct in status_pct.items():
    print(f'  {status:12s} {pct:5.1f}%')

## 5. Funding Distribution

`funding_total_usd` is the most predictive feature, but it has extreme right skew.
The log transformation is mandatory for any distance-based model (SVM, KNN, neural nets).

In [ ]:
funding_clean = df_valid['funding_total_usd'].dropna()
print(f'Rows with funding data: {len(funding_clean):,} / {len(df_valid):,}')
print(f'Median: ${funding_clean.median():,.0f}')
print(f'Mean:   ${funding_clean.mean():,.0f}  ← pulled by outliers')
print(f'Max:    ${funding_clean.max():,.0f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Linear scale — shows the problem
axes[0].hist(funding_clean.clip(upper=funding_clean.quantile(0.99)),
             bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Funding Total (linear, clipped at 99th pctl)')
axes[0].set_xlabel('USD')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))

# Log scale — interpretable
axes[1].hist(np.log1p(funding_clean), bins=50, color='teal', edgecolor='white')
axes[1].set_title('Funding Total (log1p) — After Transformation')
axes[1].set_xlabel('log(1 + USD)')

plt.tight_layout()
plt.savefig('../results/eda/funding_distribution.png', bbox_inches='tight')
plt.show()

## 6. Funding by Status (Key Predictive Signal)

This boxplot is one of the most important plots for the report.
It shows that **IPO startups raise significantly more funding** — a clear separation.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

plot_data = df_valid[df_valid['funding_total_usd'].notna()].copy()
plot_data['log_funding'] = np.log1p(plot_data['funding_total_usd'])

order = ['operating', 'acquired', 'closed', 'ipo']
sns.boxplot(data=plot_data, x='status', y='log_funding',
            order=order, palette='muted', ax=ax)
ax.set_title('Funding Total (log scale) by Startup Status')
ax.set_xlabel('Status')
ax.set_ylabel('log(1 + Funding USD)')

# Add median annotations
medians = plot_data.groupby('status')['log_funding'].median()
for i, status in enumerate(order):
    if status in medians.index:
        ax.text(i, medians[status] + 0.3, f'med={medians[status]:.1f}',
                ha='center', fontsize=8, color='darkred')

plt.tight_layout()
plt.savefig('../results/eda/funding_by_status.png', bbox_inches='tight')
plt.show()

## 7. Geographic Distribution

Two levels of analysis:
1. **Country** — USA dominates (~57%). Top 10 countries cover ~85% of data.
2. **Region** (NEW) — SF Bay Area alone is 15% and has the highest IPO rate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 15 countries
top_countries = df_valid['country_code'].value_counts().nlargest(15)
top_countries.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Top 15 Countries by Number of Startups')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Top 15 regions (NEW — not in original nb01)
top_regions = df_valid['region'].value_counts().nlargest(15)
top_regions.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Top 15 Regions by Number of Startups')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/eda/geographic_distribution.png', bbox_inches='tight')
plt.show()

# Status breakdown by top regions (key insight for report)
top5_regions = df_valid['region'].value_counts().nlargest(5).index.tolist()
df_top_reg = df_valid[df_valid['region'].isin(top5_regions)]
pivot_reg = pd.crosstab(df_top_reg['region'], df_top_reg['status'], normalize='index') * 100
print('Status distribution (%) by top 5 regions:')
print(pivot_reg.round(1).to_string())

## 8. Sector / Category Distribution

In [ ]:
df_valid['primary_category'] = (
    df_valid['category_list'].fillna('Unknown').str.split('|').str[0].str.strip()
)
top_cats = df_valid['primary_category'].value_counts().nlargest(20)

fig, ax = plt.subplots(figsize=(12, 5))
top_cats.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_title('Top 20 Primary Categories')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=55)
plt.tight_layout()
plt.savefig('../results/eda/categories.png', bbox_inches='tight')
plt.show()

# Status by top categories
top8 = df_valid['primary_category'].value_counts().nlargest(8).index.tolist()
df_top_cat = df_valid[df_valid['primary_category'].isin(top8)]
pivot_cat = pd.crosstab(df_top_cat['primary_category'], df_top_cat['status'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(12, 5))
pivot_cat.plot(kind='bar', stacked=True, ax=ax,
               color=['#2196F3', '#F44336', '#4CAF50', '#FF9800'], edgecolor='white')
ax.set_title('Status Distribution (%) by Top 8 Sectors')
ax.set_ylabel('Percentage')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Status', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../results/eda/status_by_sector.png', bbox_inches='tight')
plt.show()

## 9. Temporal Analysis

In [ ]:
df_valid['founded_year'] = df_valid['founded_at'].dt.year
yearly = df_valid.groupby('founded_year').size()
yearly = yearly[(yearly.index >= 1990) & (yearly.index <= 2023)]

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(yearly.index, yearly.values, alpha=0.4, color='steelblue')
ax.plot(yearly.index, yearly.values, color='steelblue', linewidth=2)
ax.axvline(2008, color='red', linestyle='--', alpha=0.6, label='2008 Financial Crisis')
ax.axvline(2020, color='orange', linestyle='--', alpha=0.6, label='COVID-19')
ax.legend()
ax.set_title('Number of Startups Founded per Year (1990–2023)')
ax.set_xlabel('Year')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../results/eda/founded_per_year.png', bbox_inches='tight')
plt.show()

## 10. Funding Rounds Distribution

In [ ]:
rounds = df_valid['funding_rounds'].dropna()
print(f'Median: {rounds.median():.0f} round(s)')
print(f'Max: {rounds.max():.0f} ← outlier if > ~15')
print(f'Pct with 1 round: {(rounds == 1).mean():.1%}')

fig, ax = plt.subplots(figsize=(10, 4))
rounds_capped = rounds[rounds <= 15].value_counts().sort_index()
rounds_capped.plot(kind='bar', ax=ax, color='mediumseagreen', edgecolor='white')
ax.set_title('Distribution of Funding Rounds (capped at 15)')
ax.set_xlabel('Funding Rounds')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../results/eda/funding_rounds.png', bbox_inches='tight')
plt.show()

## 11. Correlation Matrix

In [ ]:
num_df = df_valid[['funding_total_usd', 'funding_rounds']].copy()
num_df['funding_log'] = np.log1p(num_df['funding_total_usd'])
num_df['founded_year'] = df_valid['founded_year']

corr = num_df.corr()
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix (Raw Numeric Features)')
plt.tight_layout()
plt.savefig('../results/eda/correlation_matrix.png', bbox_inches='tight')
plt.show()

## Summary

**Key findings from the raw data:**

1. **Severe class imbalance:** `operating` = 79.9%, `ipo` = 2.3%. Macro F1 is the right metric, not accuracy.
2. **Funding is extremely right-skewed:** log transformation mandatory. RobustScaler recommended.
3. **IPO startups raise significantly more funding** -- clear separation in the boxplot.
4. **USA dominates** (~57%), **SF Bay Area** alone is 15% with the highest IPO rate.
5. **Biotech** has disproportionately high IPO rate -- sector is a strong predictor.
6. **founded_at missing for ~23%** -- needs careful imputation (median + indicator).
7. **Region** adds geographic signal beyond country -- SF Bay Area, NYC, London are distinct.
8. **All target labels are clean** -- no corrupted rows in the CSV (unlike the .xlsx version).

These findings directly inform every preprocessing decision in `nb02_preprocessing.ipynb`.